# 📈 Getting Started with Historical Investment Scenarios

This notebook shows you how to load, explore, and visualize scenarios from the `historical-investment-scenarios` dataset.

**Live calculator:** [fomodejavu.com](https://fomodejavu.com/)  
**Full methodology:** [fomodejavu.com/methodology](https://fomodejavu.com/methodology/)

---

## 1. Load the Master Dataset

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Load the master CSV
df = pd.read_csv('../data/scenarios-master.csv')

print(f"Total scenarios: {len(df)}")
print(f"Asset types: {df['asset_type'].unique()}")
print()
df.head(10)

## 2. Filter and Explore

In [ ]:
# Filter to stock scenarios only
stocks = df[df['asset_type'] == 'stock'].copy()

# Show top 5 by total return
top5 = stocks.nlargest(5, 'total_return_pct')[[
    'ticker', 'scenario_date', 'initial_investment_usd',
    'value_today_usd', 'total_return_pct', 'cagr_pct'
]]

top5['value_today_usd'] = top5['value_today_usd'].apply(lambda x: f'${x:,.0f}')
top5['total_return_pct'] = top5['total_return_pct'].apply(lambda x: f'{x:,.0f}%')
top5['cagr_pct'] = top5['cagr_pct'].apply(lambda x: f'{x:.1f}%')

print("Top 5 Stock Scenarios by Total Return:")
print(top5.to_string(index=False))

## 3. Visualize: $1,000 Invested — Value Today

In [ ]:
# Compare $1,000 initial investment scenarios (exclude dividend/DCA scenarios with different amounts)
compare = df[
    (df['initial_investment_usd'] == 1000) &
    (df['asset_type'].isin(['stock', 'etf']))
].copy()

compare = compare.sort_values('value_today_usd', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))

colors = ['#2563eb' if t not in ['BTC','ETH'] else '#f59e0b' for t in compare['ticker']]

bars = ax.barh(compare['ticker'], compare['value_today_usd'], color=colors, alpha=0.85)

# Add value labels
for bar, val in zip(bars, compare['value_today_usd']):
    label = f'${val/1000:.0f}K' if val < 1_000_000 else f'${val/1_000_000:.1f}M'
    ax.text(bar.get_width() + max(compare['value_today_usd']) * 0.01,
            bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=10, color='#374151')

ax.axvline(1000, color='red', linestyle='--', alpha=0.5, label='$1,000 invested')
ax.set_xlabel('Value Today (USD)', fontsize=12)
ax.set_title('$1,000 Invested — What It Would Be Worth Today\n(Historical Scenarios from historical-investment-scenarios)', fontsize=13, pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K' if x < 1_000_000 else f'${x/1_000_000:.1f}M'))
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig('../data/value_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to data/value_comparison_chart.png")

## 4. Load a Single Scenario JSON

In [ ]:
# Load a specific scenario
with open('../scenarios/stocks/amazon-ipo-1997-1000usd.json') as f:
    amzn = json.load(f)

print(f"Scenario: {amzn['title']}")
print(f"Initial investment: ${amzn['initial_investment_usd']:,}")
print(f"Purchase date: {amzn['scenario_date']}")
print(f"Price on date: ${amzn['price_on_date_usd']:.2f} (split-adjusted)")
print(f"Shares purchased: {amzn['shares_purchased']:,.1f}")
print(f"Total split multiplier: {amzn['total_split_multiplier']}×")
print(f"Split-adjusted shares today: {amzn['split_adjusted_shares']:,.0f}")
print(f"Value today: ${amzn['value_today_usd']:,.0f}")
print(f"Total return: {amzn['total_return_pct']:,.0f}%")
print(f"CAGR: {amzn['cagr_pct']:.1f}%")
print()
print("Educational note:")
print(amzn['educational_note'])
print()
print(f"Live calculator: {amzn['calculator_url']}")

## 5. Compare CAGR Across Asset Types

In [ ]:
# Box plot comparing CAGR by asset type
cagr_data = df[df['cagr_pct'].notna() & (df['cagr_pct'] != 'N/A')].copy()
cagr_data['cagr_pct'] = pd.to_numeric(cagr_data['cagr_pct'], errors='coerce')
cagr_data = cagr_data.dropna(subset=['cagr_pct'])

fig, ax = plt.subplots(figsize=(10, 6))

asset_types = cagr_data['asset_type'].unique()
data_by_type = [cagr_data[cagr_data['asset_type'] == t]['cagr_pct'].values for t in asset_types]

bp = ax.boxplot(data_by_type, labels=asset_types, patch_artist=True)

colors_map = {'stock': '#2563eb', 'crypto': '#f59e0b', 'etf': '#10b981'}
for patch, asset_type in zip(bp['boxes'], asset_types):
    patch.set_facecolor(colors_map.get(asset_type, '#6b7280'))
    patch.set_alpha(0.7)

ax.axhline(0, color='red', linestyle='--', alpha=0.4, label='Break-even')
ax.set_ylabel('CAGR (%)', fontsize=12)
ax.set_title('CAGR Distribution by Asset Type\n(All Scenarios in Dataset)', fontsize=13)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---

## Next Steps

- See `02_dividend_reinvestment.ipynb` for DRIP compounding analysis
- See `03_inflation_adjustment.ipynb` for real vs nominal return comparisons
- See `04_crypto_volatility.ipynb` for drawdown analysis
- Use the [live calculator](https://fomodejavu.com/) for custom scenarios with real-time data